# Question 1 — **(c)**, **(d)** and **(e)**

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED / "used_cars_preprocessed.csv")
print("Loaded preprocessed dataset:", df.shape)

Loaded preprocessed dataset: (5847, 12)


## (c) Change the categorical variables (“Fuel_Type” and “Transmission”) into numerical one hot encoded value.

In [2]:
df = pd.get_dummies(df, columns=["Fuel_Type", "Transmission"], dtype=int)
display(df.filter(regex="^(Fuel_Type|Transmission)_").head())

,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual
0,1,0,0,0,1
1,0,0,1,0,1
2,1,0,0,0,1
3,1,0,0,1,0
4,1,0,0,0,1


## (d) Create one more feature and add this column to the dataset (you can use mutate function in R for this). For example, you can calculate the current age of the car by subtracting “Year” value from the current year. 

In [3]:
CURRENT_YEAR = 2026
df["Car_Age"] = CURRENT_YEAR - df["Year"]
display(df[["Name", "Year", "Car_Age"]].head())

,Name,Year,Car_Age
0,Hyundai Creta 1.6 CRDi SX Option,2015,11
1,Honda Jazz V,2011,15
2,Maruti Ertiga VDI,2012,14
3,Audi A4 New 2.0 TDI Multitronic,2013,13
4,Nissan Micra Diesel XV,2013,13


## (e)  Perform select, filter, rename, mutate, arrange and summarize with group by operations (or their equivalent operations in python) on this dataset.

In [4]:
# SELECT: choose only the columns needed
selected = df[["Name", "Location", "Year", "Price"]]
display(selected.head())

,Name,Location,Year,Price
0,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,12.50
1,Honda Jazz V,Chennai,2011,4.50
2,Maruti Ertiga VDI,Chennai,2012,6.00
3,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,17.74
4,Nissan Micra Diesel XV,Jaipur,2013,3.50


In [5]:
# FILTER: cars made from 2015 onward and priced below 10 lakhs
filtered = df[(df["Year"] >= 2015) & (df["Price"] < 10)]
print("Matching cars:", len(filtered))
display(filtered[["Name", "Year", "Price"]].head())

Matching cars: 1624


,Name,Year,Price
8,Maruti Ciaz Zeta,2018,9.95
10,Maruti Swift VDI BSIV,2015,5.60
14,Honda Amaze S i-Dtech,2016,5.40
15,Maruti Swift DDiS VDI,2017,5.99
19,Maruti S Cross DDiS 200 Alpha,2015,8.25


In [6]:
# RENAME: change a column name for readability
renamed = df.rename(columns={"Kilometers_Driven": "Odometer_km"})
display(renamed[["Name", "Odometer_km"]].head())

,Name,Odometer_km
0,Hyundai Creta 1.6 CRDi SX Option,41000
1,Honda Jazz V,46000
2,Maruti Ertiga VDI,87000
3,Audi A4 New 2.0 TDI Multitronic,40670
4,Nissan Micra Diesel XV,86999


In [7]:
# MUTATE: Create new features using existing columns
df = df.assign(
    Km_Per_Year=lambda x: x["Kilometers_Driven"] / x["Car_Age"].replace(0, np.nan),
    Is_Old_Car=lambda x: x["Car_Age"] > 10
)

display(df[["Name", "Car_Age", "Km_Per_Year", "Is_Old_Car"]].head())

,Name,Car_Age,Km_Per_Year,Is_Old_Car
0,Hyundai Creta 1.6 CRDi SX Option,11,3727.272727,True
1,Honda Jazz V,15,3066.666667,True
2,Maruti Ertiga VDI,14,6214.285714,True
3,Audi A4 New 2.0 TDI Multitronic,13,3128.461538,True
4,Nissan Micra Diesel XV,13,6692.230769,True


In [8]:
# ARRANGE: sort by price from highest to lowest
sorted_cars = df.sort_values("Price", ascending=False)
display(sorted_cars[["Name", "Price"]].head())

,Name,Price
3952,Land Rover Range Rover 3.0 Diesel LWB Vogue,160.00
5620,Lamborghini Gallardo Coupe,120.00
5752,Jaguar F Type 5.0 V8 S,100.00
1457,Land Rover Range Rover Sport SE,97.07
1917,BMW 7 Series 740Li,93.67


In [9]:
# GROUP BY + SUMMARIZE: average price and number of cars per location
summary = (df.groupby("Location", as_index=False)
           .agg(Car_Count=("Price", "size"), Average_Price_Lakhs=("Price", "mean"))
           .sort_values("Average_Price_Lakhs", ascending=False))
summary["Average_Price_Lakhs"] = summary["Average_Price_Lakhs"].round(2)
display(summary)

,Location,Car_Count,Average_Price_Lakhs
3,Coimbatore,631,15.16
1,Bangalore,352,13.48
7,Kochi,640,11.31
5,Hyderabad,710,10.00
4,Delhi,540,9.88
9,Mumbai,762,9.59
0,Ahmedabad,218,8.57
2,Chennai,476,7.96
10,Pune,590,6.95
6,Jaipur,403,5.92


## Save the processed dataset

In [10]:
df.to_csv(PROCESSED / "used_cars_cleaned.csv", index=False)